In [ ]:
import os
from contextlib import closing
from pathlib import Path

import polars as pl
import psycopg2
from dotenv import load_dotenv

In [ ]:
load_dotenv()

dsn = (
  f"host=localhost dbname={os.environ["EXAMPLE_DB"]} "
  f"user={os.environ["POSTGRES_USER"]} "
  f"password={os.environ["POSTGRES_PASSWORD"]}"
)

with closing(psycopg2.connect(dsn)) as conn:
  df = (
    pl.read_database(
      "SELECT * FROM customers",
      conn,
    )
    .filter(pl.col("score") > 0)
    .group_by("country")
    .agg(pl.col("score").mean().alias("avg_score"))
    .filter(pl.col("avg_score") > 430)
    .select("country", "avg_score")
  )

print(df)

In [ ]:
with closing(psycopg2.connect(dsn)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
DROP TABLE IF EXISTS person;

CREATE TABLE IF NOT EXISTS person (
    id BIGINT NOT NULL
  , name VARCHAR(50) NULL
-- , email VARCHAR(50) NOT NULL
  , birth_date DATE NULL
  , phone VARCHAR(50) NULL
  , CONSTRAINT pk_person PRIMARY KEY (id)
)
"""
            )
    
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
ALTER TABLE person
    ADD COLUMN email VARCHAR(50) NOT NULL;
"""
            )

    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
ALTER TABLE person
    DROP COLUMN phone;
"""
            )

    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
DROP TABLE IF EXISTS person
"""
            )

In [ ]:
with closing(psycopg2.connect(dsn)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
INSERT INTO customers (id, first_name, country, score)
VALUES
    (6, 'Anna', 'USA', NULL)
  , (7, 'Sam', NULL, 100)
"""
            )
            print(cur.rowcount)

In [ ]:
with closing(psycopg2.connect(dsn)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
DROP TABLE IF EXISTS person;
CREATE TABLE person (
    id BIGINT NOT NULL
  , person_name VARCHAR(50) NOT NULL
  , birth_date DATE NULL
  , phone VARCHAR(15) NOT NULL
  , CONSTRAINT pk_person PRIMARY KEY (id)
)
"""
            )

    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
INSERT INTO person (
    id
  , person_name
  , phone
)
SELECT
    id
  , first_name
  , 'Unknown'
FROM customers
"""
            )
            print(cur.rowcount)

In [ ]:
with closing(psycopg2.connect(dsn)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
UPDATE customers
SET score = %s
WHERE id = %s
""",
                (0, 6)
            )
            print(cur.rowcount)

In [ ]:
with closing(psycopg2.connect(dsn)) as conn:
    with conn:
        with conn.cursor() as cur:
            cur.execute(
"""
DELETE FROM customers
WHERE id > %s
""",
                (5,)
            )
            print(cur.rowcount)